# Signal search: long-only, annual-rebalance portfolios (no WRDS)

A blank-slate version of the defensive max-Sharpe notebook. The engine (universe, selection, weighting, buy-and-hold returns, evaluation) is all in functions in the `signal_search/` folder next to this notebook. **To test an idea, you only edit sections B and C.**

**How to use it**
1. **Setup** (section A): run the import cell. The library lives in `signal_search/*.py`; you shouldn't need to edit it.
2. **Signals** (section B): add any new column to `crsp` (per `permno`/month).
3. **Strategies** (section C): each `Strategy` is a name, a **score function** (higher = better, NaN = ineligible), and a **weight scheme** (`'EW'`, `'IV'`, `'MV'`, `'VW'`, or your own function).
4. **Run and evaluate** (section D): screen the signals, check how they overlap, look at the design sample first, and look at the holdout only once.

**Sample:** cohorts formed each December from 1985-12 to 2024-12, so returns run January 1986 to December 2025 (40 years). **Design sample** 1986–2005, **holdout** 2006–2025.

**Data sources (no WRDS connection anywhere)**

| Input | Source |
|---|---|
| Stock panel | your local `crsp_monthly_ratios_clean.parquet` |
| FF factors + momentum | Ken French's website, downloaded once and cached to `public_data/ff_factors_monthly.csv` |
| ETF returns (optional) | local CSV (`date` + one column per ticker), or Yahoo Finance via `yfinance` if installed |
| Tickers / names (optional) | `ticker`, `comnam`, or `conm` columns in the panel if present, or a local CSV `permno,ticker,company` |

Set `USE_SYNTHETIC = True` to run everything on fake data, for example to test the code without the panel.

---
# A. Setup
The functions live in `signal_search/`: `data.py` (loaders), `signals.py` (signal helpers, sectors), `portfolio.py` (universe, selection, weights, buy-and-hold engine), `evaluate.py` (performance tables), `diagnostics.py` (screen, Fama-MacBeth, comovement), `plots.py`, `extras.py` (robustness, multi-asset, current portfolio). They are pure functions: inputs come in as arguments and nothing reads globals except defaults.

`CFG` holds every setting. The defaults are in `signal_search/config.py`; override any of them here.

In [ ]:
from signal_search import *
%matplotlib inline
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

CFG = dict(DEFAULT_CFG)          # e.g. CFG['n_hold'] = 30
pd.Series(CFG, dtype=object).to_frame('setting')

---
# B. Data and signals
`load_data` reads the panel and the factors and adds three columns: `vol36` (36-month return volatility), `mom` (12-1 momentum) and `sector` (Fama-French 12).

All per-`permno` operations work on **rows**, so a lag of 1 is the previous observation (the original notebook's convention). `add_momentum` checks for calendar gaps.

In [ ]:
USE_SYNTHETIC = False            # True = fake data, for smoke tests

crsp, ff = load_data(CFG, synthetic=USE_SYNTHETIC)
print(crsp.shape, '|', crsp['date'].min().date(), '->', crsp['date'].max().date(),
      '| factors through', ff.index.max().date())
crsp[['vol36', 'mom', 'gprof', 'debt_at', 'ep']].describe().T

### Add your own signals here
Any column you add to `crsp` is available inside score functions, for example `rank(f, 'my_signal')`. Some patterns:

In [ ]:
# crsp['vol12']   = roll(crsp, 'mthret', 12, 'std', min_periods=10)           # shorter-window vol
# crsp['mom6']    = add_momentum(crsp.copy(), skip=1, look=5)['mom']          # 6-1 momentum
# crsp['str']     = lag(crsp, 'mthret', 0)                                    # last month's return (reversal)
# crsp['maxret']  = roll(crsp, 'mthret', 12, 'max')                           # lottery-ness
# crsp['size']    = np.log(crsp['me'])
# crsp['beta36']  = ...                                                        # any custom calc, aligned to crsp rows

`data` bundles the panel, factors, and return lookups. **Rerun this cell after adding or changing signals.**

In [ ]:
data = prepare(crsp, ff)

---
# C. Strategies
A score is a function of the formation cross-section `f` (one row per stock-cohort, with all `crsp` columns plus `cohort`). It returns a Series where higher is better and NaN means ineligible.

- `form_universe` applies the screens (NYSE size percentile, price, `universe_needs`) at each formation date. The rows are one per stock-cohort and keep **every** `crsp` column, so score functions can use any signal.
- Score helpers: `rank(f, col, higher_better)` gives the percentile within the cohort. `composite(f, {...})` is the average rank (all inputs required). `mom_ok(f, pct)` is the falling-knife filter.
- `select` takes the top `n` by score, subject to a sector cap.
- A weight function has the signature `fn(sel, data, cfg) -> Series` aligned to `sel.index`. It does not need to sum to 1.

### Fixed signal set
Choose the signals before looking at any results, and don't drop signals after the fact. The book's rule is "Minimal Signal Clairvoyance" (p.341, slide 8). Trying several fixed sets is fine, but every set and every strategy you try counts toward `CFG['n_trials']`. The more signals you try, the more skeptical you should be (p.247).

`True` means higher is better.

In [ ]:
SIGNALS = {'vol36': False, 'gprof': True, 'debt_at': False, 'ep': True, 'mom': True}

The strategies from the original notebook are included as **examples and baselines**. Delete them or keep them.

In [ ]:
DEF_SPEC = {'vol36': False, 'gprof': True, 'debt_at': False, 'ep': True}

def s_lowvol(f):
  return rank(f, 'vol36', higher_better=False).where(mom_ok(f, 0.10))

def s_def(f):
  return composite(f, DEF_SPEC).where(mom_ok(f, 0.10))

BASELINES = [
  Strategy('LOWVOL_EW', s_lowvol, 'EW'),
  Strategy('DEF_EW',    s_def,    'EW'),
  Strategy('DEF_IV',    s_def,    'IV'),
  Strategy('DEF_MV',    s_def,    'MV'),     # slow-ish: one optimization per cohort
]

### Your ideas
Copy the template. Some examples follow. Each new idea is another trial, so increase `CFG['n_trials']` as you go to keep the deflated Sharpe ratio honest.

In [ ]:
# --- template ---------------------------------------------------------
# def s_my_idea(f):
#   score = composite(f, {'gprof': True, 'ep': True})             # or rank(f, 'col'), or any formula
#   return score.where(mom_ok(f, 0.10))                           # optional filters: .where(condition)
#
# def w_my_weights(sel, data, cfg):                               # optional custom weights
#   return np.sqrt(sel['me'])                                     # e.g. sqrt-cap weighting
#
# MY_IDEAS = [Strategy('MY_IDEA', s_my_idea, w_my_weights, n=30, sector_cap=0.25)]

# --- examples (comment out what you don't want) -------------------------
def s_quality_value(f):                       # drop the low-vol leg
  return composite(f, {'gprof': True, 'debt_at': False, 'ep': True}).where(mom_ok(f))

def s_def_tilted(f):                          # weighted composite: double weight on low vol
  return composite(f, DEF_SPEC, weights={'vol36': 2, 'gprof': 1, 'debt_at': 1, 'ep': 1}).where(mom_ok(f))

MY_IDEAS = [
  Strategy('QV_EW',      s_quality_value, 'EW'),
  Strategy('DEF2V_EW',   s_def_tilted,    'EW'),
  Strategy('DEF_EW_n30', s_def,           'EW', n=30),
]

STRATEGIES = BASELINES + MY_IDEAS
NAMES = [s.name for s in STRATEGIES]

---
# D. Run and evaluate
Factor values, tests and holdings are **annual**: each cohort is formed in December and held for 12 months. Returns, alphas and t-statistics use one return per holding year, because a shorter period doesn't sharpen a mean. Volatility is the one exception: it uses monthly excess returns × √12, because more observations do sharpen a variance.

### D1. Screen: rule out the obvious
Before you invest much time in a signal, run one simple full-sample test. Passing it is necessary for believing the signal, not sufficient (p.258).

The rule here: a signal **fails** if *shorting* it earns a significantly positive alpha in the design years. "Shorting it" means the bottom 20% minus the top 20% of the universe by that signal, equal-weighted and held one year. Significant means alpha > 0 and NW t > 2 (`rule='A'`). With `rule='B'`, a signal fails if the short side wins in more than half the years.

In [ ]:
screen(SIGNALS, data, CFG)

### D2. Signal overlap
**Comovement.** Each month: the rank correlation (Spearman) of every pair of signals across stocks. The table averages the 12 months of each year. The sign follows the raw columns, so for example high `vol36` against high `ep`.

In [ ]:
_, comove_by_year, _ = comove(data.crsp, list(SIGNALS), start=CFG['first_form'])
comove_by_year

**Explanatory power in combination (Fama-MacBeth).** Each December, every stock's next-year return is regressed on the percentile ranks of **all** signals together. A slope is the return gap between the top- and bottom-ranked stock, holding the other signals fixed. The t-statistics use Newey-West on the yearly slopes (design years). As a caution: with 20 pure-noise signals, about one is expected to reach t ≈ 2 by chance (p.247).

In [ ]:
fm_table, fm_slopes = fama_macbeth(SIGNALS, data, CFG)
print('average cross-sectional R2:', round(fm_table.attrs['avg R2'], 3))
fm_table

### D3. Build the portfolios
Each December, a strategy picks stocks using only data known at that date and holds them for the next 12 months. That makes every return year-ahead out-of-sample, as in the book (p.200; p.341 slide 8: "All tests are year-ahead rolling OOS"). Signal weights in composites are set in advance, not fit to returns.

Each position's value starts at `w0` and grows with its own returns over `horizon` months. A position drops out at its first missing return, and its value is redistributed pro rata.

In [ ]:
rets, hold, fu = run(STRATEGIES, data, CFG)
print('universe per cohort:', fu.groupby('cohort').size().describe()[['min', '50%', 'max']].to_dict())
print(rets.index.min().date(), '->', rets.index.max().date(), '| years:', len(rets) // 12)
weight_summary(hold, NAMES)

### D4. Design sample (1986–2005)
- **Sharpe:** the mean annual excess return divided by the annualized monthly volatility. **SE** is the iid approximation, using the number of years.
- **`z vs <bench>`:** the Jobson-Korkie/Memmel test of the Sharpe-ratio difference, on annual returns.
- **alpha:** annual returns regressed on the annual factor model in `cfg['factor_model']`, with Newey-West and OLS t-statistics.
- **max DD:** the monthly wealth path.

In [ ]:
report(rets, data, CFG, periods=('DESIGN',))

In [ ]:
design = split(rets, CFG)['DESIGN']
plot_log_growth(design, CFG)
plot_perf(design, data.ff, CFG)

**Deflated Sharpe ratio** (Bailey & López de Prado 2014; not in the book). It is the probability that a strategy's true Sharpe ratio is above the best you would expect from `n_trials` tries on noise. The **SR hurdle** is that expected best.

In [ ]:
display(dsr_table(design, data, CFG, NAMES))
name_turnover(hold, NAMES)

### D5. Quick parameter tests
Override any config value for a single run without touching `CFG`.

In [ ]:
# r30, _, _ = run([Strategy('DEF_EW_cap20', s_def, 'EW', sector_cap=0.20)], data, {**CFG, 'size_pct': 0.3})
# report(r30, data, CFG)

### D6. Holdout: look once
Pick your one or two strategies on the design sample first. Then look at the holdout (2006–2025) once: reusing out-of-sample data many times turns it into in-sample data (p.189). Report the holdout construction alongside any Sharpe ratio (p.317). The windows all end at the sample end, as in the book's fund report (p.342).

In [ ]:
# report(rets, data, CFG, periods=('HOLDOUT', '1986-2025', '2006-2025', '2016-2025'))
# plot_log_growth(rets, CFG)

### D7. Robustness: formation month and staggered cohorts
This runs the whole pipeline 12 times. `'MV'` strategies make it slow (about 500 optimizations), so pass a subset if needed.

In [ ]:
RUN_ROBUSTNESS = False
if RUN_ROBUSTNESS:
  robust_strats = [s for s in STRATEGIES if s.weight != 'MV']
  sharpe_by_fm, stag_tbl = by_formation_month(robust_strats, data, CFG)
  print('Sharpe by formation month'); display(sharpe_by_fm)
  print('Staggered portfolio'); display(stag_tbl)

### Optional: multi-asset (stock sleeve + bond/gold ETFs)
This needs ETF returns. Either save a CSV to `CFG['etf_path']` (columns `date, IEF, TLT, GLD`, monthly total returns in decimals), or run `load_etf_returns(CFG['etf_path'], ['IEF','TLT','GLD'], source='yfinance')` once to pull and cache them. If neither is available, the cell skips.

In [ ]:
STOCK_SLEEVE = 'DEF_IV'
etf_r = make_synthetic_etfs(rets.index) if USE_SYNTHETIC else load_etf_returns(CFG['etf_path'])
if etf_r is None:
  print(f"no ETF file at {CFG['etf_path']}; skipping")
elif STOCK_SLEEVE in rets:
  M, ma_tbl, W_rp = multi_asset(rets, etf_r, data, CFG, STOCK_SLEEVE)
  display(ma_tbl)
  print('Average initial weights, MULTI_RP:'); print(W_rp.mean().round(3))
  plot_log_growth(M, CFG, ['MULTI_RP', '60/40', STOCK_SLEEVE, 'EW_UNIV'], title='Multi-asset: cumulative log growth')

### Current portfolio for any strategy
This uses ticker and company columns from the panel if they exist, otherwise `CFG['names_path']`, otherwise PERMNOs only. Check tickers before trading.

In [ ]:
PICK = STRATEGIES[1]          # e.g. DEF_EW
now, f_now = current_portfolio(PICK, data, CFG, names=load_names(CFG['names_path']))
pd.set_option('display.max_rows', 60)
now.round(3)